# Split image for training and test
> Splitting image into training and test

In [ ]:
#| default_exp preprocessing.zero_degree_solder_pin.train_test_split

In [ ]:
#| hide
%load_ext autoreload
%autoreload 2


In [ ]:
#| export
import sys
from pathlib import Path
from sklearn.model_selection import train_test_split


In [ ]:
#| export
CV_TOOLS = Path(r'/home/ai_sintercra/homes/hasan/projects/git_data/cv_tools')
sys.path.append(str(CV_TOOLS))


In [ ]:
#| export
custom_lib_path = Path(r'/home/ai_warstein/homes/goni/custom_libs')
sys.path.append(str(custom_lib_path))


In [ ]:
#| export
from cv_tools.imports import *
from cv_tools.core import *
from dotenv import load_dotenv


In [ ]:
#| export
load_dotenv(dotenv_path=f'/home/ai_sintcra/homes/hasan/projects/git_data/new_test/new_test/.env')

False

In [ ]:
#| eval: false
from platform import system
if system() == 'Windows':
    core_path = Path(r'E:\CurrentTrainingData20240812_trn_val/trn_masks')
else:
    core_path = Path(r'/home/ai_easypid.work/CurrentTrainingData20240812_trn_val/trn_masks')

MODEL_NAME='ETPD-WAR1_03.keras'
mask_path = Path(
    core_path.parent, 
    f'Incoming_1B_Loetstift/Incoming_1B_Loetstift_unzip/main_im2_cropped_masks/{MODEL_NAME}/passed/masks'
    )
rect_mask_path = Path(mask_path.parent, 'rotated_rect_masks')
im_path = Path(mask_path.parent, 'images')
sn_im_path = Path(im_path.parent, 'sn_pins')
sn_msk_path = Path(im_path.parent, 'sn_masks')
sn_ov_path = Path(im_path.parent, 'sn_overlays')


In [ ]:
#| eval: false
core_path_zero = Path(core_path.parent, 'training_zero_degree_solder_pin')
trn_path = Path(core_path_zero, 'train')
val_path = Path(core_path_zero, 'val')
test_path = Path(core_path_zero, 'test')
trn_im_path = Path(trn_path, 'images')
trn_msk_path = Path(trn_path, 'masks')
val_im_path = Path(val_path, 'images')
val_msk_path = Path(val_path, 'masks')
test_im_path = Path(test_path, 'images')
test_msk_path = Path(test_path, 'masks')



# 
trn_path.mkdir(parents=True, exist_ok=True)
val_path.mkdir(parents=True, exist_ok=True)
test_path.mkdir(parents=True, exist_ok=True)
trn_im_path.mkdir(parents=True, exist_ok=True)
trn_msk_path.mkdir(parents=True, exist_ok=True)
val_im_path.mkdir(parents=True, exist_ok=True)
val_msk_path.mkdir(parents=True, exist_ok=True)
test_im_path.mkdir(parents=True, exist_ok=True)
test_msk_path.mkdir(parents=True, exist_ok=True)




In [ ]:
#| export
def parallel_copy_files(
    src_im_path,  # Source image path
    src_msk_path,  # Source mask path
    dst_im_path,  # Destination image path
    dst_msk_path,  # Destination mask path
    image_names  # List of image names to copy
    ):
    """Copy images and masks in parallel using joblib"""
    from joblib import Parallel, delayed
    from shutil import copy2
    
    def copy_single_file(im_name):
        # Copy image
        src_im = Path(src_im_path, im_name)
        dst_im = Path(dst_im_path, im_name)
        copy2(src_im, dst_im)
        
        # Copy corresponding mask
        src_msk = Path(src_msk_path, im_name)
        dst_msk = Path(dst_msk_path, im_name)
        copy2(src_msk, dst_msk)
        
        return im_name

    # Use all available CPU cores
    results = Parallel(n_jobs=-1)(delayed(copy_single_file)(im_name) for im_name in image_names)
    return results


In [ ]:
#| export
def get_synthetic_data_type(name):
    if 'rotation' in name:
        return 'rotation'
    elif 'shading' in name:
        return 'shading'
    elif 'line' in name:
        return 'line'
    else:
        return 'unknown'




In [ ]:

src_im_path =Path(mask_path.parent, 'gen_im_full')
src_msk_path =Path(mask_path.parent, 'gen_msk_full')

In [ ]:
src_im_path

Path('E:/CurrentTrainingData20240812_trn_val/Incoming_1B_Loetstift/Incoming_1B_Loetstift_unzip/main_im2_cropped_masks/ETPD-WAR1_03.keras/passed/gen_im_full')

In [ ]:
images, masks = src_im_path.ls(), src_msk_path.ls()

In [ ]:

df = pd.DataFrame()
df['fns'] = get_name_(images)
df['synthetic_data_type'] = df['fns'].apply(get_synthetic_data_type)

In [ ]:
#| export
@call_parse
def main_(
    src_im_path:Param("Source image path")='src_im_path', # Source image path
    src_msk_path:Param("Source mask path")='src_msk_path', # Source mask path
    dst_trn_im_path:Param("Destination training image path")='trn_im_path', # Destination training image path
    dst_trn_msk_path:Param("Destination training mask path")='trn_msk_path', # Destination training mask path
    dst_val_im_path:Param("Destination validation image path")='val_im_path', # Destination validation image path
    dst_val_msk_path:Param("Destination validation mask path")='val_msk_path', # Destination validation mask path
):
    """
    Split the images and masks into training and validation sets.
    """
    images, masks = Path(src_im_path).ls(), Path(src_msk_path).ls()
    df = pd.DataFrame()
    df['fns'] = get_name_(images)
    df['synthetic_data_type'] = df['fns'].apply(get_synthetic_data_type)
    trn_df, tst_df = train_test_split(df, test_size=0.2, random_state=42, stratify=df['synthetic_data_type'])
    trn_images = trn_df['fns'].tolist()
    tst_images = tst_df['fns'].tolist()

    # Copy training data
    print("Copying training data...")
    parallel_copy_files(
        src_im_path, 
        src_msk_path, 
        dst_trn_im_path, 
        dst_trn_msk_path, 
        trn_images
    )

    # Copy validation data
    print("Copying validation data...")
    parallel_copy_files(
        src_im_path, 
        src_msk_path, 
        dst_val_im_path, 
        dst_val_msk_path, 
        tst_images
    )

In [ ]:
#| export
CURRETNT_NB='/home/ai_sintcra/homes/hasan/projects/git_data/new_test/nbs'

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export('42_preprocessing.zero_degree_solder_pin.train_test_split.ipynb')

ValueError: '\\\\vihsdv140.infineon.com\\ai_sintercra\\homes\\hasan\\projects\\git_data\\nbs\\26_model_evaluation.new_model.ipynb' is not in the subpath of '\\\\vihsdv140.infineon.com\\ai_sintercra\\homes\\hasan\\projects\\git_data\\new_test\\nbs'